In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# check missing values
missings = df.isna().sum()
missings = missings.rename("count of missing values in each column")
missings = missings[missings>0]
missings

In [ ]:
# note that in info we don't have any categorical columns
# and we have alot of missing values, we cannot drop rows
df = df.fillna(0) # fill with 0
# df = df.dropna(axis=1) # drop columns with missings

In [ ]:
# check again
missings = df.isna().sum()
missings = missings.rename("count of missing values in each column")
missings = missings[missings>0]
missings

In [ ]:
# Task 2: Write your code here:
print(f"Before Removing: {df.duplicated().sum()}")
df = df.drop_duplicates() # it will affact the results
print(f"After Dropping them: {df.duplicated().sum()}")

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include="object").columns # no needed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  encoder = LabelEncoder()
  df[col] = encoder.fit_transform(df[col])

In [ ]:
# Task 5: Write your code here:
df["Target"].value_counts(normalize=True) # yes we do have imbalance data

In [ ]:
df["Target"].hist()

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns= "Target")
y= df["Target"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = CatBoostClassifier(verbose=0, n_estimators=320, max_depth=4)

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42) # no classes to stratify them, it is a reg. problem


# Storage for linear regression results for each fold
all_results = {"accuracy": [], "f1":[]}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results['accuracy'].append(accuracy)
  all_results['f1'].append(f1)

In [ ]:
  print(f"  Accuracy:  {np.mean(all_results['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
# we have more than 100 feature, I will just plot the top 10
import matplotlib.pyplot as plt

model_importance = list(zip(X.columns, model.feature_importances_))[0:10]
sorted_catboost_importance = sorted(model_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
print(f"The Most Important Feature is: {model_importance[0][0]}")
print(f"Its weight is: {model_importance[0][1]}")

In [ ]:
# Task Bonus: Write your code here:
X = df[model_importance[0][0]]

model = CatBoostClassifier(verbose=0, n_estimators=320, max_depth=4)

# Storage for linear regression results for each fold
all_results = {"accuracy": []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy += y_test == y_pred

  all_results['accuracy'].append(accuracy / len(y_test))

In [ ]:
print(f"  Accuracy:  {np.mean(all_results['accuracy']):.4f}")